# MW2 + 8 variational LSOT barycenters
Reproduce the MW2 notebook's Gaussian/synthetic/image GMM settings. Full run: 7×7 grids, image K=12, L=100, L-BFGS. Bundled image masks were recovered from notebook output; see README for provenance.

In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/LSOT_GMM')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/tmp0810/LSOT_GMM.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
%cd /content/LSOT_GMM
!pip -q install -e .

## Quick pipeline check (optional)
This uses a 3×3 grid, L=8, and 3 L-BFGS outer iterations. It checks that the pipeline works; use the next cell for reportable full settings.

In [ ]:
!python -m experiments.barycenter.run \
    --config experiments/barycenter/configs/smoke.json \
    --output-dir /content/LSOT_GMM/results/barycenter_smoke_v2

## Check optimizer convergence first
The default remains pure L-BFGS, with four shared spatial starts. A flat best-loss trace is not a convergence certificate; inspect the returned gradient and status.


In [ ]:
!python -m experiments.barycenter.run --cases gaussian synthetic --center-only \
    --output-dir /content/LSOT_GMM/results/barycenter_optimizer_check


In [ ]:
import pandas as pd
from IPython.display import display, Image
check = '/content/LSOT_GMM/results/barycenter_optimizer_check'
display(pd.read_csv(check + '/metrics.csv')[['case', 'method', 'objective_squared', 'gradient_inf', 'gradient_tolerance_met', 'status', 'total_ms']])
display(Image(filename=check + '/synthetic/center_convergence.png'))
display(Image(filename=check + '/synthetic/center_comparison.png'))


## Full experiment
All 9 methods, image K=12, L=100, 7×7 grids. Four shared starts and 80 L-BFGS steps per start; CPU default. Use a new folder for the revised solver. Completed v2 nodes can resume. The exact hard objectives remain unchanged.


In [ ]:
!python -m experiments.barycenter.run \
    --config experiments/barycenter/configs/mw2_reference.json \
    --output-dir /content/LSOT_GMM/results/barycenter_v2

In [ ]:
from IPython.display import display, Image
from pathlib import Path
root = Path('/content/LSOT_GMM/results/barycenter_v2')
methods = ['MW2'] + [f'{mode}-LSOT-{kind}' for mode in ['min', 'avg'] for kind in ['Mix', 'SMix', 'B', 'B1D']]
for case in ['synthetic', 'images']:
    for method in methods:
        print(case, method)
        display(Image(filename=str(root / case / method / 'barycenters.png')))

In [ ]:
import pandas as pd
display(pd.read_csv(root / 'summary.csv'))
# Every candidate and its input weights are saved as NPZ for further analysis.
# SW2/MW2-to-reference are agreement diagnostics, not exact W2 errors.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/barycenter_results', 'zip', root)
files.download(archive)